In [ ]:
# Set up
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
from scipy.signal import hilbert
from scipy.ndimage import gaussian_filter1d
from scipy.fft import fft, ifft
from tqdm import tqdm


In [ ]:
# # Hilbert Envelope Analysis

# Directory
stimuli_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/Stimuli'
wav_files = [f for f in os.listdir(stimuli_dir) if f.endswith('.wav')]

for wav_file in wav_files:
    path = os.path.join(stimuli_dir, wav_file)

    # Load audio
    audio, sr = librosa.load(path, sr=None)

    # Hilbert envelope
    analytic = hilbert(audio)
    envelope = np.abs(analytic)
    envelope = gaussian_filter1d(envelope, sigma=100)  # heavy smoothing for display

    # Normalize both
    audio_norm = audio / np.max(np.abs(audio))
    envelope_norm = envelope / np.max(envelope)

    # Time axis
    t = np.linspace(0, len(audio) / sr, len(audio))

    # Plot
    plt.figure(figsize=(10, 3))
    plt.plot(t, audio_norm, color='lightgray', linewidth=0.8, label='Speech waveform')  # raw waveform in gray
    plt.plot(t, envelope_norm, color='orange', linewidth=2, label='Envelope')  # clean orange line
    plt.title(f'Broadband Envelope – {wav_file}', fontsize=12)
    plt.xlabel('Time (s)')
    plt.ylabel('Normalized Amplitude')
    plt.ylim(-1.05, 1.05)
    plt.grid(False)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Average of 1st Dev of Hilbert envelope


# Path to your stimuli directory
stimuli_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/Stimuli'

# List available wav files
wav_files = [f for f in os.listdir(stimuli_dir) if f.endswith('.wav')]

# Pick the first wav file
audio_path = os.path.join(stimuli_dir, wav_files[0])
print("Using file:", audio_path)

# Load the audio file
audio, sr = librosa.load(audio_path, sr=None)

# Compute Hilbert envelope
analytic_signal = hilbert(audio)
envelope = np.abs(analytic_signal)

# Apply Gaussian smoothing
envelope_smooth = gaussian_filter1d(envelope, sigma=100)

# Time vector
t = np.linspace(0, len(audio)/sr, len(audio))

# Numerical derivative
dt = 1 / sr
derivative_np = np.gradient(envelope_smooth, dt)

# Savitzky-Golay smoothed derivative
window_length = 101
polyorder = 3
derivative_savgol = savgol_filter(envelope_smooth, window_length, polyorder, deriv=1, delta=dt)

# Plot
plt.figure(figsize=(12, 6))

plt.subplot(2, 1, 1)
plt.plot(t, envelope_smooth, label='Smoothed Envelope')
plt.title('Envelope')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(t, derivative_np, label='Numerical Derivative (gradient)')
plt.plot(t, derivative_savgol, label='Smoothed Derivative (Savitzky-Golay)', linestyle='--')
plt.title('Envelope Derivative')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude change rate')
plt.legend()

plt.tight_layout()
plt.show()



In [ ]:
# Morlet Wavelet Analysis

### STEP 1 — Extract envelope from audio

# Directory containing .wav files
stimuli_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/Stimuli'
wav_files = [f for f in os.listdir(stimuli_dir) if f.endswith('.wav')]

# Process all stimuli
stimuli_envelopes = {}
for wav_file in wav_files:
    path = os.path.join(stimuli_dir, wav_file)

    # Load audio
    audio, sr_audio = librosa.load(path, sr=None)

    # Hilbert transform to get envelope
    analytic = hilbert(audio)
    envelope = np.abs(analytic)

    # Smooth envelope (optional)
    envelope_smooth = gaussian_filter1d(envelope, sigma=100)

    # Normalize envelope
    envelope_norm = envelope_smooth / np.max(envelope_smooth)

    # Store for later
    stimuli_envelopes[wav_file] = (envelope_norm, sr_audio)

    # Plot example
    plt.figure(figsize=(10, 3))
    plt.plot(audio/np.max(np.abs(audio)), label='Waveform')
    plt.plot(envelope_norm, color='orange', label='Envelope')
    plt.title(f'Envelope – {wav_file}')
    plt.legend()
    plt.show()


### STEP 2 — Resample envelope to EEG sampling rate

# EEG sampling rate
sr_eeg = 512

# Example: resample 1st stimulus
example_file = wav_files[0]
envelope, sr_audio = stimuli_envelopes[example_file]
envelope_resampled = librosa.resample(envelope, orig_sr=sr_audio, target_sr=sr_eeg)

# Make sure envelope length matches EEG length for alignment
# (this assumes you know the duration matches EEG segment)


### STEP 3 — Extract time-frequency features from EEG using Morlet

# Load your EEG data (example: 88 channels, 10 sec recording)
EEG = np.random.randn(81, sr_eeg * 10)  # Replace this with your real EEG

# Morlet parameters
freqs = np.logspace(np.log10(100), np.log10(10000), 200)
n_cycles = 6

n_channels, n_samples = EEG.shape
n_frequencies = len(freqs)
power = np.zeros((n_channels, n_frequencies, n_samples))

# FFT convolution
n_conv = n_samples + len(EEG[0,:]) - 1
EEG_fft = fft(EEG, n=n_conv, axis=1)

for f_idx, f in tqdm(enumerate(freqs), total=len(freqs), desc="Morlet transform"):

    wavetime = np.arange(-1, 1, 1/sr_eeg)
    sigma = n_cycles / (2 * np.pi * f)
    wavelet = np.exp(2j * np.pi * f * wavetime) * np.exp(-wavetime**2 / (2 * sigma**2))
    wavelet = wavelet / np.sqrt(sigma * np.sqrt(np.pi))

    wavelet_fft = fft(wavelet, n=n_conv)

    for ch in range(n_channels):
        convolution_result = ifft(EEG_fft[ch, :] * wavelet_fft)
        convolution_result = convolution_result[:n_samples]
        power[ch, f_idx, :] = np.abs(convolution_result) ** 2


### STEP 4 — Align EEG power & envelope

# Now your EEG power and envelope_resampled are both time-aligned
# You can analyze e.g.:
# - Correlations
# - TRF models
# - Decoding
# - Phase-locking

# Example: plot time-frequency power for 1 channel
plt.figure(figsize=(10, 5))
plt.imshow(power[0,:,:], aspect='auto',
           extent=[0, n_samples/sr_eeg, freqs[0], freqs[-1]],
           origin='lower', cmap='jet')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Time-Frequency Power (Channel 0)')
plt.colorbar(label='Power')
plt.show()
